In [1]:
import json
import warnings
import numpy as np
import pandas as pd
import optuna
import torch
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality

from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader
# =========================
# 1) Config & Hyperparameters
# =========================
SEED = 121
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_PATH = r"C:\Users\NFU_Power_Lab\Desktop\DAT_ResearchArea\BestAwardAI2026\random-split\wpt_train_random.csv"
OUT_AUG_PATH = "wpt_train_random_aug_tabddpm.csv"          
OUT_SYN_ONLY_PATH = "wpt_train_random_syn_only_tabddpm.csv" 
OPTUNA_PARAMS_PATH = "best_tabddpm_params.json"             

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Balancing Policy
MAX_SYN_MULTIPLIER = 2.0 
TARGET_MINIMUM = 50      
SYN_PER_TYPE_CAP = 300    

# Optuna Config
OPTUNA_TRIALS = 20  # TabDDPM chạy khá nặng, 20 trials là đủ
FINAL_EPOCHS = 2000 # TabDDPM dùng số vòng lặp (iterations) cao hơn GAN

def snap_load_to_grid(df, col="load", step=0.1, lo=0.5, hi=3.0):
    """Round load to fixed grid and clip to allowed range."""
    out = df.copy()
    out[col] = pd.to_numeric(out[col], errors="coerce")
    out[col] = np.round(out[col] / step) * step
    out[col] = out[col].clip(lo, hi)
    return out

def knn_filter_by_type(
    real_df,
    syn_df,
    feature_cols=("load", "efficiency"),
    type_col="type",
    k=5,
    keep_quantile=0.95,
    min_real_per_type=10,
):
    """
    Keep synthetic samples that are close to real manifold, per type.
    Method:
      - Fit scaler + kNN on REAL samples of each type
      - For each synthetic sample of that type, compute mean distance to k nearest real neighbors
      - Keep those with distance <= per-type threshold (quantile)
    """
    real_df = real_df.copy()
    syn_df = syn_df.copy()
    real_df[type_col] = real_df[type_col].astype(str)
    syn_df[type_col] = syn_df[type_col].astype(str)

    kept_parts = []
    for t in sorted(real_df[type_col].unique()):
        real_t = real_df[real_df[type_col] == t]
        syn_t = syn_df[syn_df[type_col] == t]
        if len(syn_t) == 0:
            continue

        # If too few real samples for that type, skip filtering (or keep all)
        if len(real_t) < min_real_per_type:
            kept_parts.append(syn_t)
            continue

        Xr = real_t.loc[:, feature_cols].to_numpy(dtype=float)
        Xs = syn_t.loc[:, feature_cols].to_numpy(dtype=float)

        # Scale using REAL stats of that type
        scaler = StandardScaler()
        Xr_s = scaler.fit_transform(Xr)
        Xs_s = scaler.transform(Xs)

        # kNN on real
        nn = NearestNeighbors(n_neighbors=min(k, len(real_t)), metric="euclidean")
        nn.fit(Xr_s)

        dists, _ = nn.kneighbors(Xs_s, return_distance=True)
        # distance score = mean distance to k neighbors
        score = dists.mean(axis=1)

        # per-type threshold
        thr = np.quantile(score, keep_quantile)
        mask = score <= thr

        kept = syn_t.loc[mask].copy()
        kept_parts.append(kept)

    if kept_parts:
        return pd.concat(kept_parts, ignore_index=True)
    return syn_df.iloc[0:0].copy()

def make_balanced_augmented_train(df_train, syn_df, seed=121,
                                 target_mode="p75", cap_per_type=300):
    """
    Build an augmented training set by balancing types using synthetic samples.
    - df_train: real train
    - syn_df: synthetic pool generated by TabDDPM
    """
    rng = np.random.default_rng(seed)

    # Ensure correct dtypes
    df_train = df_train.copy()
    syn_df = syn_df.copy()
    df_train["type"] = df_train["type"].astype(str)
    syn_df["type"] = syn_df["type"].astype(str)

    # Decide target count per type
    counts = df_train["type"].value_counts()
    if target_mode == "max":
        target = int(counts.max())
    elif target_mode == "median":
        target = int(counts.median())
    elif target_mode == "p75":
        target = int(np.percentile(counts.values, 75))
    else:
        raise ValueError("target_mode must be max/median/p75")

    parts = [df_train.assign(is_synthetic=0)]
    for t, n_real in counts.items():
        need = target - int(n_real)
        if need <= 0:
            continue
        need = min(need, cap_per_type)

        pool_t = syn_df[syn_df["type"] == t]
        if len(pool_t) == 0:
            continue

        # sample from pool
        take = pool_t.sample(n=min(need, len(pool_t)), random_state=seed).copy()
        take["is_synthetic"] = 1
        parts.append(take)

    aug = pd.concat(parts, ignore_index=True)
    return aug


def objective(trial):
    # ---- TabDDPM hyperparams ----
    n_iter = trial.suggest_int("n_iter", 500, 1500, step=500)
    batch_size = trial.suggest_categorical("batch_size", [128, 256])
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)

    # ---- Train TabDDPM on df_train only ----
    synthesizer = Plugins().get(
        "ddpm",
        n_iter=n_iter,
        batch_size=batch_size,
        lr=lr,
        device=DEVICE
    )

    try:
        synthesizer.fit(GenericDataLoader(df_train))
        # ---- Generate synthetic POOL ----
        pool_size = min(max(2000, 2 * len(df_train)), 10000)
        syn_pool = synthesizer.generate(pool_size).dataframe()

        # Basic sanitation FIRST
        syn_pool["type"] = syn_pool["type"].astype(str)
        syn_pool["load"] = pd.to_numeric(syn_pool["load"], errors="coerce")
        syn_pool["efficiency"] = pd.to_numeric(syn_pool["efficiency"], errors="coerce")
        syn_pool = syn_pool.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)

        # 1) Snap load to 0.1 grid
        syn_pool = snap_load_to_grid(
            syn_pool,
            col="load",
            step=0.1,
            lo=float(df_train["load"].min()),
            hi=float(df_train["load"].max())
        )

        # Clip efficiency to real train range
        eff_min, eff_max = float(df_train["efficiency"].min()), float(df_train["efficiency"].max())
        syn_pool["efficiency"] = syn_pool["efficiency"].clip(eff_min, eff_max)

        # 2) kNN filter (per type)
        syn_pool = knn_filter_by_type(
            real_df=df_train,
            syn_df=syn_pool,
            feature_cols=("load", "efficiency"),
            type_col="type",
            k=5,
            keep_quantile=0.95,
            min_real_per_type=10
        )

        # (optional) drop any NaNs introduced (usually none)
        syn_pool = syn_pool.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)

        # ---- Build balanced augmented train ----
        aug_train = make_balanced_augmented_train(
            df_train=df_train,
            syn_df=syn_pool,
            seed=SEED,
            target_mode="p75",
            cap_per_type=300
        )

        # ---- Train & eval TWO classifiers (LR + RF) on augmented train ----
        X_train = aug_train[["load", "efficiency"]].values
        y_train = aug_train["type"].values

        X_val = df_val[["load", "efficiency"]].values
        y_val = df_val["type"].values
        # 1) Logistic Regression (balanced)
        lr_clf = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])
        lr_clf.fit(X_train, y_train)
        pred_lr = lr_clf.predict(X_val)
        f1_lr = f1_score(y_val, pred_lr, average="macro")
        # 2) Random Forest (balanced)
        rf_clf = Pipeline([
            ("scaler", StandardScaler()),
            ("model", RandomForestClassifier(
                n_estimators=400,
                random_state=SEED,
                class_weight="balanced",
                n_jobs=-1
            ))
        ])
        rf_clf.fit(X_train, y_train)
        pred_rf = rf_clf.predict(X_val)
        f1_rf = f1_score(y_val, pred_rf, average="macro")

        score = 0.5 * (f1_lr + f1_rf)
        # (Optional) log each metric for inspection in Optuna
        trial.set_user_attr("f1_lr", float(f1_lr))
        trial.set_user_attr("f1_rf", float(f1_rf))

        # # (Optional) track quality as a secondary metric
        # syn_val = synthesizer.generate(len(df_val)).dataframe()
        # quality_report = evaluate_quality(real_data=df_val, synthetic_data=syn_val, metadata=metadata, verbose=False)
        # trial.set_user_attr("sdv_quality", float(quality_report.get_score()))

        syn_val = synthesizer.generate(len(df_val)).dataframe()
        syn_val["type"] = syn_val["type"].astype(str)
        syn_val["load"] = pd.to_numeric(syn_val["load"], errors="coerce")
        syn_val["efficiency"] = pd.to_numeric(syn_val["efficiency"], errors="coerce")
        syn_val = syn_val.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)
        syn_val = snap_load_to_grid(
            syn_val,
            col="load",
            step=0.1,
            lo=float(df_train["load"].min()),
            hi=float(df_train["load"].max())
        )
        eff_min, eff_max = float(df_train["efficiency"].min()), float(df_train["efficiency"].max())
        syn_val["efficiency"] = syn_val["efficiency"].clip(eff_min, eff_max)
        quality_report = evaluate_quality(
            real_data=df_val,
            synthetic_data=syn_val,
            metadata=metadata,
            verbose=False
        )
        trial.set_user_attr("sdv_quality", float(quality_report.get_score()))

        return float(score)

    except Exception as e:
        return 0.0

# =========================
# 2) Load & Preprocess Data
# =========================
df = pd.read_csv(TRAIN_PATH)
expected = {"type", "load", "efficiency"}
if expected - set(df.columns):
    raise ValueError(f"Missing columns: {expected - set(df.columns)}")

df["type"] = df["type"].astype(str)
df["load"] = pd.to_numeric(df["load"], errors="coerce")
df["efficiency"] = pd.to_numeric(df["efficiency"], errors="coerce")
df = df.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)

print("Train shape:", df.shape)

val_ratio = 0.2
df_train, df_val = train_test_split(
    df,
    test_size=val_ratio,
    random_state=SEED,
    shuffle=True,
    stratify=df["type"]
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
print("Train/Val split:")
print("  df_train:", df_train.shape, "df_val:", df_val.shape)
print("  Type counts (train):\n", df_train["type"].value_counts().sort_index())
print("  Type counts (val):\n", df_val["type"].value_counts().sort_index())

# Metadata cho SDV (để đánh giá Optuna)
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df)
metadata.update_column(column_name='type', sdtype='categorical')
metadata.update_column(column_name='load', sdtype='numerical')
metadata.update_column(column_name='efficiency', sdtype='numerical')

# =========================
# 4) Run Optuna Optimization
# =========================
print(f"\n=== Bắt đầu tìm kiếm Optuna ({OPTUNA_TRIALS} Trials) ===")
study = optuna.create_study(direction="maximize") 
study.optimize(objective, n_trials=OPTUNA_TRIALS)

print("\n=== Optuna Hoàn Thành ===")
best_params = study.best_params
print(f"“Best utility score (avg macro-F1 LR+RF)”: {study.best_value:.4f}")
print(f"Best parameters: {best_params}")

with open(OPTUNA_PARAMS_PATH, 'w') as f:
    json.dump(best_params, f, indent=4)

# =========================
# 5) Train Final TabDDPM Model
# =========================
print(f"\n=== Bắt đầu huấn luyện mô hình TabDDPM cuối cùng ===")
final_synthesizer = Plugins().get(
    "ddpm",
    n_iter=FINAL_EPOCHS, # Dùng số lần lặp lớn nhất cho mô hình chốt
    batch_size=best_params["batch_size"],
    lr=best_params["lr"],
    device=DEVICE
)

# Train trên toàn bộ dữ liệu
final_loader = GenericDataLoader(df)
final_synthesizer.fit(final_loader)
print("Huấn luyện Diffusion thành công!")

# =========================
# 6) Generate Synthetic Data (Over-generation & Filtering)
# =========================
counts = df["type"].value_counts().sort_index()
print("\nCounts by type (before):\n", counts)

syn_parts = []
eff_min, eff_max = df["efficiency"].min(), df["efficiency"].max()

# Sinh trước 10,000 mẫu để tạo "Bể chứa" dữ liệu
print("\nĐang sinh 10,000 mẫu từ không gian Diffusion để lọc...")
pool_df = final_synthesizer.generate(10000).dataframe()
# Sanitize
pool_df["type"] = pool_df["type"].astype(str)
pool_df["load"] = pd.to_numeric(pool_df["load"], errors="coerce")
pool_df["efficiency"] = pd.to_numeric(pool_df["efficiency"], errors="coerce")
pool_df = pool_df.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)
# 1) Snap load
pool_df = snap_load_to_grid(
    pool_df,
    col="load",
    step=0.1,
    lo=float(df["load"].min()),
    hi=float(df["load"].max())
)
# Clip efficiency
pool_df["efficiency"] = pool_df["efficiency"].clip(eff_min, eff_max)
# 2) kNN filter vs REAL df (or df_train if you prefer)
pool_df = knn_filter_by_type(
    real_df=df,          # or df_train for stricter "train-only" manifold
    syn_df=pool_df,
    feature_cols=("load", "efficiency"),
    type_col="type",
    k=5,
    keep_quantile=0.95,
    min_real_per_type=10
)

for t in sorted(df["type"].unique()):
    n_real = int((df["type"] == t).sum())

    if t == 'K' or n_real > 300:
        continue

    desired_target = max(TARGET_MINIMUM, int(counts.median()))
    need = desired_target - n_real
    max_allowed_syn = int(n_real * MAX_SYN_MULTIPLIER)
    need = min(need, max_allowed_syn)
    need = min(need, SYN_PER_TYPE_CAP)
 
    if need <= 0:
        continue

    print(f"Đang trích xuất {need} mẫu giả cho loại vật thể: {t}...")
    
    # Trích xuất từ Bể dữ liệu
    class_pool = pool_df[pool_df["type"] == t]
    
    if len(class_pool) >= need:
        syn_df = class_pool.sample(n=need, random_state=SEED)
    else:
        # Nếu Bể không đủ, liên tục sinh thêm các mẻ 2000 mẫu cho đến khi đủ
        print(f"  -> Không đủ mẫu trong pool, đang khởi động lại Diffusion...")
        collected = [class_pool]
        while sum(len(x) for x in collected) < need:
            extra_df = final_synthesizer.generate(2000).dataframe()

            extra_df["type"] = extra_df["type"].astype(str)
            extra_df["load"] = pd.to_numeric(extra_df["load"], errors="coerce")
            extra_df["efficiency"] = pd.to_numeric(extra_df["efficiency"], errors="coerce")
            extra_df = extra_df.dropna(subset=["type", "load", "efficiency"]).reset_index(drop=True)

            extra_df = snap_load_to_grid(
                extra_df,
                col="load",
                step=0.1,
                lo=float(df["load"].min()),
                hi=float(df["load"].max())
            )
            extra_df["efficiency"] = extra_df["efficiency"].clip(eff_min, eff_max)

            # (optional but recommended) apply kNN filter vs real
            extra_df = knn_filter_by_type(
                real_df=df,
                syn_df=extra_df,
                feature_cols=("load", "efficiency"),
                type_col="type",
                k=5,
                keep_quantile=0.95,
                min_real_per_type=10
            )

            collected.append(extra_df[extra_df["type"] == t])
        syn_df = pd.concat(collected).head(need)
    
    syn_df["is_synthetic"] = 1
    syn_parts.append(syn_df)

# =========================
# 7) Merge & Save
# =========================
if syn_parts:
    syn_df_all = pd.concat(syn_parts, ignore_index=True)
else:
    syn_df_all = pd.DataFrame(columns=["type","load","efficiency","is_synthetic"])

real_df = df.copy()
real_df["is_synthetic"] = 0
aug_df = pd.concat([real_df, syn_df_all], ignore_index=True)

before = df["type"].value_counts().sort_index()
after = aug_df["type"].value_counts().sort_index()
report = pd.DataFrame({"before": before, "after": after}).fillna(0).astype(int)

report.to_csv("tabddpm_type_counts_before_after.csv", encoding="utf-8-sig")
aug_df.to_csv(OUT_AUG_PATH, index=False, encoding="utf-8-sig")

if not syn_df_all.empty:
    syn_df_all.to_csv(OUT_SYN_ONLY_PATH, index=False, encoding="utf-8-sig")

print("\nGenerated synthetic rows:", len(syn_df_all))
print("Saved augmented train:", OUT_AUG_PATH)
if not syn_df_all.empty:
    print("Saved synthetic-only data:", OUT_SYN_ONLY_PATH)
print("\nCounts by type (after):\n", after)

# =========================
# 8) Xuất File Thống kê Phân phối
# =========================
stats_before = df.groupby("type")["efficiency"].agg(["mean","std","count"]).rename(columns={"count":"n_before"})
stats_after  = aug_df.groupby("type")["efficiency"].agg(["mean","std","count"]).rename(columns={"count":"n_after"})
stats_merge = stats_before.join(stats_after, lsuffix="_real", rsuffix="_aug")
stats_merge.to_csv("tabddpm_type_stats_before_after.csv", encoding="utf-8-sig")
print("Saved: tabddpm_type_stats_before_after.csv")

[KeOps] Warning : No C++ compiler found. Define CXX environment variable or install g++.
[KeOps] Warning : No C++ compiler found. You need to either define the CXX environment variable pointing to a valid compiler, or ensure that 'g++' is installed and in your PATH.
[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
[KeOps] Warning : No C++ compiler found. You need to either define the CXX environment variable pointing to a valid compiler, or ensure that 'g++' is installed and in your PATH.
[KeOps] Warning : No C++ compiler available to check for OpenMP support.
[KeOps] Warning : OpenMP support is not available. Disabling OpenMP.
Train shape: (1310, 3)
Train/Val split:
  df_train: (1048, 3) df_val: (262, 3)
  Type counts (train):
 type
A     18
B     71
C     19
D     76
E     73
F     21
G     76
H    223
I     18
J    148
K    305
Name: count, dtype: int64
  Type counts (val):
 type
A     4
B    18
C     5
D    19
E    18
F     5
G    19
H    56

[2026-03-12T00:17:27.533377+0800][33272][CRITICAL] load failed: Cannot find DGL C++ graphbolt library at c:\Users\NFU_Power_Lab\anaconda3\envs\SOH\Lib\site-packages\dgl\graphbolt\graphbolt_pytorch_2.10.0.dll
[2026-03-12T00:17:27.534376+0800][33272][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_goggle' has no attribute 'plugin'
[2026-03-12T00:17:27.535380+0800][33272][CRITICAL] module plugin_goggle load failed
Epoch: 100%|██████████| 1500/1500 [01:09<00:00, 21.50it/s, loss=0.344]
[2026-03-12T00:19:08.543478+0800][33272][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_goggle' has no attribute 'plugin'
[2026-03-12T00:19:08.544478+0800][33272][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_goggle' has no attribute 'plugin'
[2026-03-12T00:19:08.545478+0800][33272][CRITICAL] module plugin_goggle load failed
Epoch: 100%|██████████| 1500/1500 [01:09<00:00, 21.62it/s, loss=0.353]
[2026-03-12T00:20:49.484009+0800][33272][CRITICAL] load failed: m


=== Optuna Hoàn Thành ===
“Best utility score (avg macro-F1 LR+RF)”: 0.3744
Best parameters: {'n_iter': 1000, 'batch_size': 128, 'lr': 0.002270242264818564}

=== Bắt đầu huấn luyện mô hình TabDDPM cuối cùng ===


Epoch: 100%|██████████| 2000/2000 [02:54<00:00, 11.47it/s, loss=0.338]


Huấn luyện Diffusion thành công!

Counts by type (before):
 type
A     22
B     89
C     24
D     95
E     91
F     26
G     95
H    279
I     23
J    185
K    381
Name: count, dtype: int64

Đang sinh 10,000 mẫu từ không gian Diffusion để lọc...
Đang trích xuất 44 mẫu giả cho loại vật thể: A...
Đang trích xuất 2 mẫu giả cho loại vật thể: B...
Đang trích xuất 48 mẫu giả cho loại vật thể: C...
Đang trích xuất 52 mẫu giả cho loại vật thể: F...
Đang trích xuất 46 mẫu giả cho loại vật thể: I...
  -> Không đủ mẫu trong pool, đang khởi động lại Diffusion...

Generated synthetic rows: 192
Saved augmented train: wpt_train_random_aug_tabddpm.csv
Saved synthetic-only data: wpt_train_random_syn_only_tabddpm.csv

Counts by type (after):
 type
A     66
B     91
C     72
D     95
E     91
F     78
G     95
H    279
I     69
J    185
K    381
Name: count, dtype: int64
Saved: tabddpm_type_stats_before_after.csv
